# 💼 AI Portfolio Advisory Copilot — LangChain version

A multi-agent system that manages an **Alpaca paper-trading portfolio** through the **Alpaca MCP server**.
It combines SEC 10-K research (Module 3) with technical, risk and portfolio-management agents (Module 4),
checks every proposal against the investor's mandate, and only sends orders after **your approval**.
Everything runs on a **local LLM** (`nemotron-3.5-lightning:30b-mlx` via Ollama).

```
          Alpaca paper account (via MCP)                       data/10k PDFs
     account · positions · clock · daily bars · news       (Apple, Alphabet, Tesla)
                 │                                                   │
                 ▼                                                   ▼
  ┌─────────────────────────────────── for each stock ───────────────────────────────────┐
  │  🔎 Research analyst   10-K search + Alpaca news (MCP)        → FundamentalView        │
  │  📈 Technical analyst  indicators + Alpaca snapshot (MCP)     → TechnicalView          │
  │  ⚠️ Risk manager       views + portfolio + investor profile   → RiskView               │
  │  👔 Portfolio manager  everything above                        → TradeProposal (BUY/SELL/HOLD, shares) │
  └───────────────────────────────────────────────────────────────────────────────────────┘
                 │
                 ▼
  🛡️ Compliance rule engine (plain Python): suitability · concentration · cash buffer · order size · no shorting
                 │
                 ▼
  👤 Human approval  ──►  place_stock_order (MCP, PAPER account)  ──►  📒 audit trail (JSONL)
```

### Design principles
| Principle | How |
|---|---|
| **LLMs propose, code disposes** | Agents only *propose* trades; `check_trade()` blocks unsuitable trades and resizes oversized ones |
| **Least privilege** | Agents get two read-only MCP tools (`get_stock_snapshot`, `get_news`); orders are placed by Python, never by an agent |
| **Paper only** | `ALPACA_PAPER_TRADE=true` is forced in code (`alpaca_env()`), whatever `.env` says |
| **Human in the loop** | Dry run by default; with `EXECUTE_ORDERS = True` every order still needs a typed `yes` |
| **Prompt-injection aware** | News is treated as untrusted data (the Alpaca server also wraps it in a trust-boundary envelope) |
| **Measured, not assumed** | A fixed evaluation set scores the agents; results are saved to compare frameworks |

Shared, framework-agnostic code (data, prompts, schemas, rules, evaluation) lives in `portfolio_core.py`; this notebook
contains only the LangChain-specific parts, so the two versions can be compared line by line.

**LangChain pieces used:** `ChatOllama` (local model), `langchain-mcp-adapters` (Alpaca MCP tools), `create_agent` + `ToolStrategy` for tool-using agents with validated outputs, `with_structured_output` for the others.

## ⚙️ Setup

1. **Alpaca paper account** — sign up at [alpaca.markets](https://alpaca.markets), switch to *Paper Trading* and create
   API keys. Copy `.env.example` to `.env` in this folder and fill in:
   ```
   ALPACA_API_KEY=PK...
   ALPACA_SECRET_KEY=...
   ```
2. **uv** (runs the Alpaca MCP server with `uvx alpaca-mcp-server`): `curl -LsSf https://astral.sh/uv/install.sh | sh`
3. **Ollama models**:
   ```bash
   ollama pull nemotron-3.5-lightning:30b-mlx
   ollama pull nomic-embed-text
   OLLAMA_CONTEXT_LENGTH=16384 ollama serve     # a bigger context window helps with long prompts
   ```
4. `pip install -r requirements.txt`, then run the notebook top to bottom.

> ⚠️ Educational project on a **paper** account. Not investment advice.

In [ ]:
import asyncio, shutil, time
from contextlib import AsyncExitStack

import pandas as pd
import portfolio_core as core
from portfolio_core import (UNIVERSE, PROFILES, AGENT_MCP_TOOLS, MARKET_DATA_FEED, PortfolioSnapshot, TenKIndex,
                            FundamentalView, TechnicalView, RiskView, TradeProposal, AuditLog,
                            RESEARCH_PROMPT, TECHNICAL_PROMPT, RISK_PROMPT, PM_PROMPT,
                            research_request, technical_request, risk_request, pm_request,
                            alpaca_env, unwrap, bars_by_symbol, compute_indicators, check_trade, build_order,
                            decisions_table, run_evaluation, compare_frameworks)

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

llm = ChatOllama(
    model="nemotron-3.5-lightning:30b-mlx",
    temperature=0.1
)
audit = AuditLog("langchain")
print("✅ Model: nemotron-3.5-lightning:30b-mlx (Ollama) | audit log:", audit.path)

## 1️⃣ Connect to the Alpaca paper account through MCP

The **Model Context Protocol (MCP)** lets an application use tools exposed by a separate server process. Here
`uvx alpaca-mcp-server` starts Alpaca's official server (stdio transport) with our paper keys, and we keep one
connection open for the whole notebook. We use it two ways:

* **Direct calls from Python** (`call(...)`) for deterministic steps: account, positions, bars, orders.
* **As agent tools** — only the read-only `get_stock_snapshot` and `get_news`.

In [ ]:
mcp_client = MultiServerMCPClient({
    "alpaca": {"transport": "stdio", "command": shutil.which("uvx") or "uvx",
               "args": ["alpaca-mcp-server"], "env": alpaca_env()},
})

stack = AsyncExitStack()                       # keeps one MCP session open across notebook cells
session = await stack.enter_async_context(mcp_client.session("alpaca"))


async def call(tool_name: str, args: dict = None):
    """Call an Alpaca MCP tool directly (no LLM) and return plain Python data."""
    return unwrap(await session.call_tool(tool_name, args or {}))


mcp_tools = await load_mcp_tools(session)      # every Alpaca tool as a LangChain tool
print(f"✅ Connected to Alpaca MCP (paper) - {len(mcp_tools)} tools, e.g. {[t.name for t in mcp_tools][:8]}")

# The agents only see these read-only tools:
agent_mcp_tools = [t for t in mcp_tools if t.name in AGENT_MCP_TOOLS]

In [ ]:
account = await call("get_account_info")
positions = await call("get_all_positions")
clock = await call("get_clock")
snapshot = PortfolioSnapshot.from_alpaca(account, positions, clock)

if not snapshot.account_number.startswith("PA"):
    print("⚠️ Account number does not look like a paper account - double-check your keys!")
print(f"Account {snapshot.account_number} (paper)\n{snapshot.describe()}")

## 2️⃣ Choose the investor profile (the mandate)

The profile is the client's mandate — the same `low / medium / high` risk tolerance as the customer database in
Module 3. It drives both the agents' instructions and the hard rules in the compliance engine.

In [ ]:
PROFILE = PROFILES["moderate"]          # "conservative", "moderate" or "aggressive"
print(PROFILE.describe())

## 3️⃣ Market data → technical indicators

We download ~4 months of daily bars for the research universe through MCP (IEX feed, free for paper accounts) and
compute RSI, moving averages, MACD, 20-day return and annualised volatility **in Python** — LLMs are unreliable at
arithmetic, so they receive the numbers instead of computing them.

In [ ]:
bars = bars_by_symbol(await call("get_stock_bars", {"symbols": ",".join(UNIVERSE), "timeframe": "1Day",
                                                    "days": 120, "feed": MARKET_DATA_FEED, "limit": 1000}))
indicators = {sym: compute_indicators(bars[sym]) for sym in UNIVERSE}
pd.DataFrame(indicators).T[["last_close", "trend", "rsi14", "return_20d_pct", "volatility_annual", "sma20", "sma50"]]

## 4️⃣ 10-K search (RAG)

The 2024 annual reports in `data/10k/` are split into chunks and embedded locally with `nomic-embed-text`.
Embeddings are cached in `.cache/`, so only the first run takes a few minutes.

In [ ]:
tenk = TenKIndex().build()
print(tenk.search("TSLA", "supply chain risks", k=1)[:400], "...")

## 5️⃣ The agents

| Agent | Tools | Output |
|---|---|---|
| Research analyst | `search_10k` (local) + `get_news` (MCP) | `FundamentalView` |
| Technical analyst | `get_stock_snapshot` (MCP) | `TechnicalView` |
| Risk manager | – | `RiskView` |
| Portfolio manager | – | `TradeProposal` |

Prompts and output schemas come from `portfolio_core.py` (identical in both notebooks).

In [ ]:
@tool
def search_10k(symbol: str, question: str) -> str:
    """Search the company's 2024 Form 10-K (ticker AAPL, GOOGL or TSLA) and return the most relevant passages with page numbers."""
    return tenk.search(symbol, question)


# Tool-using agents: ReAct loop + validated final answer
research_agent = create_agent(llm, tools=[search_10k, *agent_mcp_tools], system_prompt=RESEARCH_PROMPT,
                              response_format=ToolStrategy(FundamentalView))
technical_agent = create_agent(llm, tools=agent_mcp_tools, system_prompt=TECHNICAL_PROMPT,
                               response_format=ToolStrategy(TechnicalView))
# No tools needed: a single structured call is enough
risk_chain = llm.with_structured_output(RiskView)
pm_chain = llm.with_structured_output(TradeProposal)


async def ask_agent(agent, prompt: str):
    state = await agent.ainvoke({"messages": [{"role": "user", "content": prompt}]})
    return state["structured_response"]


async def ask_chain(chain, system_prompt: str, prompt: str):
    return await chain.ainvoke([("system", system_prompt), ("human", prompt)])

## 6️⃣ Analyse the portfolio

For each stock: research and technical analysis run **in parallel**, then the risk manager, then the portfolio
manager, then the deterministic `check_trade()` rules. Every step is written to the audit trail.

> ⏱️ With a local 30B model expect roughly 1–3 minutes per stock.

In [ ]:
async def analyze_symbol(symbol: str) -> dict:
    start, ind = time.perf_counter(), indicators[symbol]
    fundamental, technical = await asyncio.gather(                   # independent -> run in parallel
        ask_agent(research_agent, research_request(symbol)),
        ask_agent(technical_agent, technical_request(symbol, ind)),
    )
    risk = await ask_chain(risk_chain, RISK_PROMPT, risk_request(symbol, PROFILE, snapshot, ind, fundamental, technical))
    proposal = await ask_chain(pm_chain, PM_PROMPT, pm_request(symbol, PROFILE, snapshot, ind, fundamental, technical, risk))
    compliance = check_trade(symbol, proposal, risk, PROFILE, snapshot, ind)
    for event, obj in [("fundamental", fundamental), ("technical", technical), ("risk", risk),
                       ("proposal", proposal), ("compliance", compliance)]:
        audit.log(event, symbol, obj)
    print(f"✓ {symbol}: {proposal.action} {proposal.quantity} -> {compliance.final_action} {compliance.final_qty}")
    return {"symbol": symbol, "fundamental": fundamental, "technical": technical, "risk": risk,
            "proposal": proposal, "compliance": compliance, "seconds": time.perf_counter() - start}


decisions = [await analyze_symbol(sym) for sym in UNIVERSE]
decisions_table(decisions)

In [ ]:
for r in decisions:
    c = r["compliance"]
    print(f"\n{'=' * 90}\n{r['symbol']}: proposed {c.proposed_action} {c.proposed_qty} -> final {c.final_action} {c.final_qty}")
    print(f"🔎 {r['fundamental'].outlook}: {r['fundamental'].summary}")
    print(f"📈 {r['technical'].signal} ({r['technical'].confidence:.0%}): {r['technical'].rationale}")
    print(f"⚠️ {r['risk'].risk_level}: {r['risk'].rationale}")
    print(f"👔 {r['proposal'].rationale}")
    for ch in c.checks:
        print(f"   {'✅' if ch.passed else '⛔'} {ch.name}: {ch.detail}")

## 7️⃣ Human approval → paper orders

`EXECUTE_ORDERS = False` (default) only prints the orders. Set it to `True` to be asked, order by order, whether to
send it to your **paper** account via the MCP `place_stock_order` tool (market order, `day`, with an idempotent
`client_order_id`). When the market is closed the order is queued for the next open.

In [ ]:
EXECUTE_ORDERS = False

for r in decisions:
    c = r["compliance"]
    if not c.approved:
        print(f"{c.symbol}: HOLD - no order")
        continue
    order = build_order(c)
    print(f"\n📝 {c.symbol}: {order['side'].upper()} {order['qty']} shares at market (~${c.final_qty * c.price:,.0f})")
    print(f"   why: {r['proposal'].rationale}")
    if not EXECUTE_ORDERS:
        print("   dry run - set EXECUTE_ORDERS = True to send it to the paper account")
        continue
    if input("   Send this order to the Alpaca PAPER account? (yes/no): ").strip().lower() != "yes":
        audit.log("order_rejected_by_human", c.symbol, order)
        print("   ❌ skipped")
        continue
    result = await call("place_stock_order", order)
    audit.log("order_submitted", c.symbol, {"order": order, "result": result})
    print("   ✅ submitted:", result.get("status") if isinstance(result, dict) else result)

In [ ]:
# Latest orders and refreshed positions from the paper account
orders = await call("get_orders", {"status": "all", "limit": 10})
cols = ["submitted_at", "symbol", "side", "qty", "type", "status", "filled_avg_price"]
display(pd.DataFrame(orders)[[c for c in cols if c in pd.DataFrame(orders).columns]] if orders else "no orders yet")
snapshot = PortfolioSnapshot.from_alpaca(await call("get_account_info"), await call("get_all_positions"), await call("get_clock"))
print(snapshot.describe())

## 8️⃣ Evaluation

Eight fixed scenarios (synthetic inputs, no market data needed) with expected outcomes — e.g. *a conservative
investor must not buy a 65%-volatility stock*, *a held stock turning bearish should be sold*, *no short selling*.
Each scenario runs the risk manager → portfolio manager → rules.

| Metric | Meaning |
|---|---|
| `system_accuracy` | final decision (after the rules) is the expected one |
| `llm_accuracy` | the LLM's raw proposal was already correct **and** compliant |
| `rule_interventions` | how often the rule engine had to block or resize a proposal |
| `errors` | schema / parsing failures |

Results are saved to `outputs/eval_<framework>.json`; run both notebooks to compare frameworks.

In [ ]:
async def run_risk(prompt: str) -> RiskView:
    return await ask_chain(risk_chain, RISK_PROMPT, prompt)


async def run_pm(prompt: str) -> TradeProposal:
    return await ask_chain(pm_chain, PM_PROMPT, prompt)


eval_df = await run_evaluation(run_risk, run_pm, framework="langchain")
eval_df[["scenario", "expected", "llm_proposal", "final", "llm_ok", "system_ok", "seconds"]]

In [ ]:
compare_frameworks()

In [ ]:
await stack.aclose()      # stop the Alpaca MCP server
print("MCP connection closed")

## ✅ Summary

* **Alpaca MCP** — `MultiServerMCPClient.session()` + `load_mcp_tools` (agent tools) and `session.call_tool` (direct calls)
* **Agents** — two `create_agent` ReAct agents + two `with_structured_output` chains; parallel with `asyncio.gather`
* **Safety** — paper trading forced, read-only agent tools, deterministic rule engine, human approval, audit trail
  (`outputs/audit_langchain.jsonl`)
* **Evaluation** — `outputs/eval_langchain.json`; `compare_frameworks()` puts both frameworks side by side

**Ideas to extend:** a scheduled daily run, more tickers (add their 10-K PDFs to `data/10k/`), bracket orders with
stop-loss / take-profit (`place_stock_order` supports them), or rebalancing towards target weights.